In [1]:
################### Processing csv files to get points values! ########

from datetime import datetime, timedelta
import os
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import requests
import sys
import geopandas as gpd
import pandas as pd
from shapely.geometry import MultiPoint
from shapely.geometry import Point
import fiona
import contextily as ctx
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
import simplekml

In [2]:
path = '/Users/bz294/Documents/WhiteZone/Points/'
file1 = 'RevisedTimeSeries2__.csv'
df = pd.read_csv(path + file1)
df_filtered = df[~df['system:index'].str.contains('_015043_')]
#df_filtered = df

coord = pd.read_csv('/Users/bz294/Documents/WhiteZone/Points/points_coord2.csv')

# Display the first few rows of the DataFrame
print(coord.head())

#df_filtered = pd.merge(df_filtered, coord[['ID', 'lat', 'lon']], on='ID', how='left') ## no need to merge


  descriptio     lat     lon  ID
0         WZ  25.253 -80.600   1
1         WZ  25.262 -80.542   2
2         WZ  25.262 -80.567   3
3         TZ  25.257 -80.486   4
4         WZ  25.263 -80.590   5


In [3]:
unique_names = sorted(df_filtered['ID'].unique())
unique_name_series = {}
unique_name_indices = {}

for name in unique_names:
    # Find the indices where 'name' matches the current unique value
    indices = df_filtered.index[df_filtered['ID'] == name].tolist()
    # Store the indices in the dictionary with the unique name as the key
    unique_name_indices[name] = indices
    name_data = df_filtered[df_filtered['ID'] == name][['lat', 'lon', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'ndmi', 'ndvi', 'savi', 'descriptio']].to_numpy()
    unique_name_series[name] = name_data

In [4]:
unique_dates = sorted(df_filtered['date'].unique())
unique_dates_series = {}
unique_dates_indices = {}

for name in unique_dates:
    # Find the indices where 'name' matches the current unique value
    indices = df_filtered.index[df_filtered['date'] == name].tolist()
    # Store the indices in the dictionary with the unique name as the key
    unique_dates_indices[name] = indices
    dates_data = df_filtered[df_filtered['date'] == name][['lat', 'lon', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'ndmi', 'ndvi', 'savi', 'descriptio']].to_numpy()
    unique_dates_series[name] = dates_data

In [5]:
keys = list(unique_dates_series.keys())
#values_2013_04_02 = unique_dates_series[keys[0]]

date_vector = [datetime.strptime(date, '%Y-%m-%d') for date in keys]
date_vector

start_date = datetime(2023, 2, 1)
end_date = datetime(2023, 4, 30)

# Get indices for dates within the specified range
indices_in_range = [i for i, date in enumerate(date_vector) if start_date <= date <= end_date]

selected_keys = [keys[i] for i in indices_in_range]

# Step 3: Retrieve data from unique_dates_series using the selected keys
training_data = [unique_dates_series[key] for key in selected_keys]



In [6]:
predictor = []
target = []

# Loop through each (1585, 10) array in training_data

for array in training_data:
    # Get rows where the target is not 'TZ'
    mask = array[:, 11] != 'TZ'
    
    # Apply mask to select only rows without 'TZ' in the target column
    predictor.append(array[mask, 2:11])  # First 9 columns
    target.append(array[mask, 11])       # Target column

# Convert to NumPy arrays if desired
predictor = np.array(predictor)
target = np.array(target)

In [7]:
# Reshape predictors and target variables for compatibility with scikit-learn
X = predictor.reshape(-1, 9)  # Shape: (5 * 1585, 9)
y = target.reshape(-1)        # Shape: (5 * 1585,)

# Initialize the RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Fit the model to the reshaped data
model.fit(X, y)

# After training, the model can be used for predictions or evaluation
y_pred = model.predict(X)

# Calculate accuracy by comparing the predictions with the true labels (y)
accuracy = model.score(X, y)

print(f"Training accuracy: {accuracy * 100:.2f}%")

Training accuracy: 93.96%


In [8]:
############# Testing using 2013 data ################

start_date = datetime(2013, 2, 1)
end_date = datetime(2013, 4, 30)

# Get indices for dates within the specified range
indices_in_range = [i for i, date in enumerate(date_vector) if start_date <= date <= end_date]

selected_keys = [keys[i] for i in indices_in_range]

# Step 3: Retrieve data from unique_dates_series using the selected keys
testing_data = [unique_dates_series[key] for key in selected_keys]

predictor_test = []
target_test = []
# Loop through each (1585, 10) array in training_data
for array in testing_data:

    mask = array[:, 11] != 'TZ'
    # Slice the first 9 columns and append to first_9_columns list
    predictor_test.append(array[mask, 2:11])  # First 9 columns
    target_test.append(array[mask, 11])       # Target column

# Convert to NumPy arrays if desired
predictor_test = np.array(predictor_test)
target_test = np.array(target_test)

In [ ]:
selected_keys

In [9]:
############ make predictions ########

X = predictor_test.reshape(-1, 9)  # Shape: (5 * 1585, 9)
y = target_test.reshape(-1)        # Shape: (5 * 1585,)

y_pred_test = model.predict(X)

# Calculate accuracy by comparing the predictions with the true labels (y)
accuracy = model.score(X, y)

print(f"Training accuracy: {accuracy * 100:.2f}%")

Training accuracy: 77.13%


In [10]:
predictor_test.shape

(3, 1411, 9)

In [11]:
target_test.shape

(3, 1411)

In [ ]:
len(testing_data) * testing_data[0].shape[0]

In [12]:
###### know the indices of the wrong predictions ####

y = y.reshape(target_test.shape[0], predictor_test.shape[1])
y_pred_test = y_pred_test.reshape(target_test.shape[0], predictor_test.shape[1])

# Find where predictions don't match the true labels
incorrect_mask = y_pred_test != y

# Get the indices of incorrect predictions for each sample
incorrect_indices = np.argwhere(incorrect_mask) ## these are the indices from the subset without TZ, not all!!

# Calculate accuracy
accuracy = model.score(X, y.reshape(-1))
print(f"Training accuracy: {accuracy * 100:.2f}%")

# Print incorrect indices
print("Indices of incorrect predictions (in 5 x 1371 format):")
print(incorrect_indices)

Training accuracy: 77.13%
Indices of incorrect predictions (in 5 x 1371 format):
[[   0   58]
 [   0  103]
 [   0  111]
 ...
 [   2 1215]
 [   2 1402]
 [   2 1407]]


In [15]:
###### get lat and lon for "incorrect" predictions #######
lat = []
lon = []
select_scene = 0
first = training_data[0]
mask = first[:, 11] != 'TZ'
filtered_first = first[mask]

# Apply mask to select only rows without 'TZ' in the target column
lon.append(filtered_first[:, 1])       # First 9 columns
lat.append(filtered_first[:, 0])       # Target column

lat = np.array(lat).flatten()
lon = np.array(lon).flatten()
filtered_indices = incorrect_indices[incorrect_indices[:, 0] == select_scene, 1] ## select the third one, which is March, 2013 according to GE. 
filtered_indices = np.array(filtered_indices)
#lat[filtered_indices]
#lon[filtered_indices]
#target_test[0, filtered_indices] -- ground truth
#y_pred_test[0, filtered_indices] -- predicted 

In [ ]:
incorrect_indices

In [ ]:
y_pred_test[0, :]

In [16]:
######### generating kml ######################

kml = simplekml.Kml()

# Define colors for each 'Description' value
colors = {
    'VZ': 'ff00ff00',  # Green in KML color format (aabbggrr)
    'WZ': 'ff0000ff',  # Red
}

values = y_pred_test[0, filtered_indices] 
for lattemp, lontemp, val in zip(lat[filtered_indices], lon[filtered_indices], y_pred_test[select_scene, filtered_indices]): ## pay attention to change the index here to 2. 
    color = colors.get(val, 'ff000000')  # Default to black if value not in colors
    pnt = kml.newpoint(coords=[(lontemp, lattemp)])  # KML uses (longitude, latitude)
    pnt.style.iconstyle.color = color  # Set the color based on the value

# Save the KML file
kml.save("/Users/bz294/Documents/WhiteZone/Points/KML/model_check2_2013.kml")

In [ ]:
values

In [ ]:
filtered_indices

In [ ]:
len(lat[0])

In [ ]:
name_1_data = unique_name_series.get(1)

# Display the data
type(name_1_data)
name_1_indices = unique_name_indices.get(1)
len(name_1_indices)

In [ ]:
##### Get time stamps #######
date_values = df_filtered.loc[name_1_indices, 'date'].values

np.unique(date_values).shape
date_values
date_times = np.array([datetime.strptime(date, '%m/%d/%y') for date in date_values])

days_ind = (date_times - date_times[0]).astype('timedelta64[D]').astype(int)

# Convert the first element to 0 and keep the rest
days_ind[0] = 0

# Display the result
print(days_ind)

In [ ]:
date_times

In [ ]:
date_strings

In [ ]:
########### Plot time series of a random pixel ####################
%matplotlib inline
b2 = name_1_data[:, 0] 
ndvi = name_1_data[:, 7] 

plt.figure(figsize=(10, 7))
plt.plot(days_ind, ndvi, marker='o', linestyle='-')
plt.title('Time Series Plot')
plt.xlabel('Date')
plt.ylabel('Value')
xticks = range(0, max(days_ind) + 1, 500)

date_strings = [date_times[0] + timedelta(days=500 * i) for i in range(9)]
date_strings = [date.strftime('%Y-%m-%d') for date in dates]

plt.xticks(ticks=xticks, labels=date_strings, rotation=45)

plt.grid(True)
plt.tight_layout()  # Adjust layout to prevent clipping of labels
plt.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.2)
plt.show()

In [ ]:
############# model building ###############




In [ ]:
xticks_indices

In [ ]:
len(df)

In [ ]:
unique_dates

In [ ]:
50100 / 981

In [ ]:
################ Plot time series for selected pixels ####################

name_1_data = unique_name_series.get(1)

In [ ]:
unique_dates